[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/15_mlp.ipynb)

# 🟠 Medium: SwiGLU MLP

Реализуйте **SwiGLU MLP** — gated feed-forward блок современных language models. В отличие от обычного `Linear → activation → Linear`, здесь две параллельные projections создают gate и content, после чего перемножаются поэлементно.

$$\text{SwiGLU}(x) = \text{down\_proj}\big(\text{SiLU}(\text{gate\_proj}(x)) \odot \text{up\_proj}(x)\big)$$

where $\text{SiLU}(x) = x \cdot \sigma(x)$

### Формы по шагам
Для входа `(..., d_model)` обе верхние projections дают `(..., d_ff)`. `SiLU(gate)` и `up` имеют одинаковую shape, их Hadamard product остаётся `(...,d_ff)`. `down_proj` возвращает последнюю dimension обратно в `d_model`; все leading dimensions сохраняются.

### Роль gate
`up_proj(x)` несёт candidate content, а `SiLU(gate_proj(x))` задаёт мягкий input-dependent коэффициент пропускания для каждой hidden coordinate. Gate не ограничен строго диапазоном `[0,1]`, потому что SiLU умножает sigmoid на сам input.

### Контракт
```python
class SwiGLUMLP(nn.Module):
    def __init__(self, d_model: int, d_ff: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Требования
- наследуйтесь от `nn.Module`;
- `gate_proj` и `up_proj`: `nn.Linear(d_model,d_ff)`;
- `down_proj`: `nn.Linear(d_ff,d_model)`;
- activation: SiLU/Swish через `F.silu` или `x * sigmoid(x)`.

### План реализации
1. Создайте три Linear submodules с правильными directions.
2. Независимо вычислите gate и up branches.
3. Примените SiLU только к gate branch.
4. Перемножьте branches поэлементно и примените down projection.

### Быстрые самопроверки
- shapes трёх weight matrices соответствуют контракту;
- 2-D и 3-D inputs сохраняют leading dimensions;
- forward совпадает с явным `down(silu(gate(x)) * up(x))`;
- gradients получают input и все parameters.

### Типичные ошибки
- SiLU применяется после перемножения branches;
- gate и up складываются вместо elementwise multiplication;
- down projection имеет перепутанные input/output dimensions;
- используется matrix multiplication между branches;
- один Linear ошибочно переиспользуется для gate и up.

### Официальные материалы и примеры
- [`torch.nn.SiLU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.SiLU.html) — formula, shape и пример activation;
- [`torch.nn.Linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) — projection semantics и shapes;
- [`torch.nn.GLU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GLU.html) — родственная gated architecture для сравнения.

### Вопросы для самопроверки
1. Почему gate и content должны иметь одинаковую hidden shape?
2. Чем SiLU отличается от sigmoid gate?
3. Почему down projection возвращает `d_model`?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class SwiGLUMLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        pass  # Initialize gate_proj, up_proj, down_proj

    def forward(self, x):
        pass  # down_proj(silu(gate_proj(x)) * up_proj(x))

In [ ]:
# 🧪 Debug
mlp = SwiGLUMLP(d_model=64, d_ff=128)
x = torch.randn(2, 8, 64)
out = mlp(x)
print("Output shape:", out.shape)  # (2, 8, 64)
print("Params:", sum(p.numel() for p in mlp.parameters()))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('mlp')